In [ ]:
import sys
import subprocess

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "pyspark", "datasets", "wordcloud", "mlflow"])
    from google.colab import drive
    drive.mount("/content/drive")

import os
if IN_COLAB:
    os.chdir("/content/drive/MyDrive/amazon-reviews-sentiment-analysis")
else:
    # Ensure src/ is importable
    sys.path.insert(0, os.path.abspath("..")) if os.path.basename(os.getcwd()) == "notebooks" else None

# Scalability Analysis

In [ ]:
from src.config import ProjectConfig, FeatureConfig
from src.data_loader import get_spark_session, load_data
from src.models import build_full_pipeline
from src.evaluation import compute_metrics
from src.utils import timer

config = ProjectConfig()
spark = get_spark_session(config.spark)
train_df, test_df = load_data(spark, config.data)
train_df.cache()
test_df.cache()
full_count = train_df.count()
print(f"Full training set: {full_count:,} rows")

## Training Time vs Dataset Size

In [ ]:
fractions = [0.01, 0.05, 0.1, 0.25, 0.5, 0.75, 1.0]
scalability_results = []

for frac in fractions:
    sample = train_df.sample(fraction=frac, seed=42)
    n_rows = sample.count()
    print(f"\n--- Fraction: {frac} ({n_rows:,} rows) ---")

    pipeline = build_full_pipeline("count_vectorizer", "logistic_regression", config.features)

    with timer(f"Training on {n_rows:,} rows") as elapsed:
        model = pipeline.fit(sample)
    training_time = elapsed()

    predictions = model.transform(test_df)
    metrics = compute_metrics(predictions)

    scalability_results.append({
        "fraction": frac,
        "n_rows": n_rows,
        "training_time": training_time,
        "accuracy": metrics["accuracy"],
        "throughput": n_rows / training_time,
    })
    print(f"Accuracy: {metrics['accuracy']:.4f} | Time: {training_time:.1f}s | Throughput: {n_rows/training_time:,.0f} rows/sec")

## Results Visualization

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

df = pd.DataFrame(scalability_results)

fig, ax1 = plt.subplots(figsize=(10, 6))
ax2 = ax1.twinx()

ax1.plot(df["n_rows"], df["training_time"], "b-o", lw=2, markersize=8, label="Training Time")
ax2.plot(df["n_rows"], df["accuracy"], "r-s", lw=2, markersize=8, label="Accuracy")

ax1.set_xlabel("Dataset Size (rows)")
ax1.set_ylabel("Training Time (seconds)", color="blue")
ax2.set_ylabel("Accuracy", color="red")

ax1.tick_params(axis="y", labelcolor="blue")
ax2.tick_params(axis="y", labelcolor="red")

ax1.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x/1e6:.1f}M" if x >= 1e6 else f"{x/1e3:.0f}K"))

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc="center right")

plt.title("Scalability: Training Time & Accuracy vs Dataset Size")
plt.tight_layout()
plt.savefig("docs/scalability.png", dpi=150, bbox_inches="tight")
plt.show()

## Analysis

In [ ]:
print("Throughput Analysis:")
print(df[["fraction", "n_rows", "training_time", "accuracy", "throughput"]].to_string(index=False))
print(f"\nPeak throughput: {df['throughput'].max():,.0f} rows/sec")
print(f"Accuracy at 1%: {df.iloc[0]['accuracy']:.4f}")
print(f"Accuracy at 100%: {df.iloc[-1]['accuracy']:.4f}")
print(f"Accuracy gain from 25% to 100%: {df.iloc[-1]['accuracy'] - df[df['fraction']==0.25].iloc[0]['accuracy']:.4f}")

In [ ]:
spark.stop()